In [0]:
%run ../utils

In [0]:
def convert_roman(roman_str):
    """
    Doc String
    """
    if not roman_str:
        return None
    try:
        return roman.fromRoman(roman_str)
    except:
        return None
    
roman_udf = udf(convert_roman, IntegerType())

In [0]:
nat_dex_df = spark.table(f"{STAGING_DATABASE_PREFIX}.nat_dex")

national_pokedex_df = (
    nat_dex_df
    .withColumn("pokemon_entry", explode(col("pokemon_entries")))
    .select(
        col("id").alias("pokedex_id"),
        col("name").alias("pokedex_name"),
        col("pokemon_entry.entry_number").alias("pokedex_number"),
        col("pokemon_entry.pokemon_species.name").alias("pokemon_name")
    )
)

national_pokedex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.national_pokedex")

In [0]:
species_df = spark.table(f"{STAGING_DATABASE_PREFIX}.species")

national_pokedex_species_df = (
    species_df
    .withColumn("generation_name", split(col("generation.name"), '-')[1])
    .withColumn("generation", roman_udf(species_df["generation_name"]))
    .select(
        col("id").alias("pokedex_number"),
        col("name").alias("pokemon_name"),
        col("generation"),
        col("is_legendary"),
        col("is_mythical"),
        col("evolves_from_species.name").alias("evolves_from"),
    )
)

national_pokedex_species_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.species")

In [0]:
varieties_df = spark.table(f"{STAGING_DATABASE_PREFIX}.varieties")

national_pokedex_varieties_df = (
    varieties_df
    .withColumn("hp", expr("filter(stats, x -> x.stat.name = 'hp')[0].base_stat"))
    .withColumn("attack", expr("filter(stats, x -> x.stat.name = 'attack')[0].base_stat"))
    .withColumn("defence", expr("filter(stats, x -> x.stat.name = 'defense')[0].base_stat"))
    .withColumn("special_attack", expr("filter(stats, x -> x.stat.name = 'special-attack')[0].base_stat"))
    .withColumn("special_defence", expr("filter(stats, x -> x.stat.name = 'special-defense')[0].base_stat"))
    .withColumn("speed", expr("filter(stats, x -> x.stat.name = 'speed')[0].base_stat"))
    .withColumn("type_1", expr("CASE WHEN size(filter(types, x -> x.slot = 1)) > 0 THEN filter(types, x -> x.slot = 1)[0].type.name ELSE NULL END"))
    .withColumn("type_2", expr("CASE WHEN size(filter(types, x -> x.slot = 2)) > 0 THEN filter(types, x -> x.slot = 2)[0].type.name ELSE NULL END"))
    .select(
        col("id").alias('pokeapi_id'),
        col("name").alias('pokemon_name'),
        col("base_pokedex_number"),
        col("base_name").alias("base_pokemon_name"),
        col("is_default"),
        col("hp"),
        col("attack"),
        col("defence"),
        col("special_attack"),
        col("special_defence"),
        col("speed"),
        col("type_1"),
        col("type_2")
    )
)

national_pokedex_varieties_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.varieties")